# Face Recognition 
The model is divided in functional units...
1. save_face(name, number_of_faces_to_capture) => capture a person's face and save it in provided 'name' directory inside faces directory.
2. save_faces_to_csv() =>  saves all faces as csv in 'generated_dataset' directory and deletes face data. (concats with previous dataset if exist)
3. model_fit() => train model with LinearSVC
4. predict() => recognise face on the camera

In [1]:
#capture and save face data of a person
import shutil
import cv2
import os
def save_face(name,number_of_faces):
    """Pass name of the face and provide number of faces to save"""
    
    #creating directory to store faces
    try:
        os.mkdir(f'faces/{name}')
    except FileExistsError:
        shutil.rmtree(f'faces/{name}')
        os.mkdir(f'faces/{name}')
    # video class
    vdo=cv2.VideoCapture(0)
    #face detection model
    model=cv2.CascadeClassifier('pre_trained_model\haarcascade_frontalface_default.xml')

    total_face=0
    #capturing faces
    while True:
        flag,img=vdo.read()  # returns first image from camera/video
        faces=model.detectMultiScale(img) # returns all face's bounded box (x,y,w,h) in a list

        #drawing rectangle and saving faces from an image
        for x,y,w,h in faces:
            cv2.rectangle(img,(x,y),(x+h,y+w),(255,255,255),1)
            extracted_face=img[y:y+h,x:x+w]
            cv2.imwrite(f'faces/{name}/{total_face}.png',extracted_face)
            total_face+=1
            
        cv2.putText(img,f'Faces Captured:{total_face}',(0,30),cv2.FONT_HERSHEY_COMPLEX,1,(255,255,255),1)
        cv2.putText(img,'Press c to exit',(0,60),cv2.FONT_HERSHEY_COMPLEX,1,(255,255,255),1)
        
        cv2.imshow('camera',img)
        key=cv2.waitKey(50)
        if key==ord('c') or total_face>=number_of_faces:
            break
    cv2.destroyAllWindows()
    vdo.release()

In [2]:
save_face('shariq',50)

In [3]:
# to save image data as csv
import shutil
import os
import cv2
import numpy as np
import pandas as pd
def save_faces_to_csv():
    directories=os.listdir('faces')
    df=pd.DataFrame()
    for dir in directories:
        names=os.listdir(f'faces/{dir}')
        for name in names:
            img=cv2.imread(f'faces/{dir}/{name}',cv2.IMREAD_GRAYSCALE) # return 2d array of image (grayscale image)
            img=cv2.resize(img,(150,150))   # to make all faces having same no. of pixel 
            vector=img.reshape(1,22500)  
            df1=pd.DataFrame(vector,columns=[str(i) for i in range(22500)])
            df1['output']=dir
            df=pd.concat([df,df1],axis=0, ignore_index=True)

        
        #now to delete axisting image data because we have saved to csv
        shutil.rmtree(f'faces/{dir}')
    
    # saing to csv
    try:
        previous_df=pd.read_csv('generated_dataset/face.csv')
        final_to_save=pd.concat([previous_df,df],axis=0,ignore_index=True)
        final_to_save.to_csv('generated_dataset/face.csv',index=False)
    except:
        df.to_csv('generated_dataset/face.csv',index=False)    

In [4]:
save_faces_to_csv()

In [5]:
# train model on the available data (saved csv)
import pandas as pd
from sklearn.linear_model import LogisticRegression   # aditya sir trained his model with this algo that is why i am training.  
def model_fit():
    global model
    df=pd.read_csv('generated_dataset/face.csv')
    X=df.iloc[:,:-1].values
    y=df.iloc[:,-1].values
    model=LogisticRegression(max_iter=10000)
    model.fit(X,y)

In [6]:
model_fit()

In [7]:
#to predict
import cv2
def predict():
    vdo=cv2.VideoCapture(0)
    while True:
        flag,img=vdo.read()
        img_gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
        CascadeClassifierModel=cv2.CascadeClassifier('pre_trained_model\haarcascade_frontalface_default.xml')
        faces=CascadeClassifierModel.detectMultiScale(img)
        for x,y,w,h in faces:
            cv2.rectangle(img,(x,y),(x+h,y+w),(255,255,255),1)
            extracted_face=img_gray[y:y+h,x:x+w]
            extracted_face=cv2.resize(extracted_face,(150,150))
            vector=extracted_face.reshape((1,22500))
            pred=model.predict(vector)
            cv2.putText(img,pred[0],(x,y),cv2.FONT_HERSHEY_COMPLEX,1,(255,255,255),1)
        cv2.putText(img,'Press c to exit',(0,60),cv2.FONT_HERSHEY_COMPLEX,1,(255,255,255),1)
        cv2.imshow('Face Recognition Model',img)
        key=cv2.waitKey(50)
        if key==ord('c'):
            break
    cv2.destroyAllWindows()
    vdo.release()

In [8]:
predict()